Task 1 The Tower of Hanoi



In [9]:

import time
from z3 import *


def build(n, m, k, idle=False, symmetry=True):
    on = [[[Bool(f"on_{d}_{w}_{t}") for t in range(k + 1)]
           for w in range(m)] for d in range(n)]
    obj = [[Bool(f"obj_{d}_{t}") for t in range(k)] for d in range(n)]
    to = [[Bool(f"to_{w}_{t}") for t in range(k)] for w in range(m)]
    C = []

    # ---- initial / final state -------------------------------------------
    for d in range(n):
        for w in range(m):
            C.append(on[d][w][0] if w == 0 else Not(on[d][w][0]))
            C.append(on[d][w][k] if w == m - 1 else Not(on[d][w][k]))

    for t in range(k):
        any_obj = Or([obj[d][t] for d in range(n)])
        any_to = Or([to[w][t] for w in range(m)])

        # ---- one move per step (or none, if idle steps are allowed) -------
        C.append(AtMost(*[obj[d][t] for d in range(n)], 1))
        C.append(AtMost(*[to[w][t] for w in range(m)], 1))
        if idle:
            C.append(any_obj == any_to)          # idle <=> no destination
            if t + 1 < k:                        # idle steps only at the end
                C.append(Implies(Or([obj[d][t + 1] for d in range(n)]), any_obj))
        else:
            C.append(any_obj)
            C.append(any_to)

        for d in range(n):
            for w in range(m):
                smaller_on_w = [on[e][w][t] for e in range(d)]
                if smaller_on_w:
                    # precondition I : d must be on top of its source tower
                    C.append(Implies(And(obj[d][t], on[d][w][t]),
                                     Not(Or(smaller_on_w))))
                    # precondition II: no smaller disk on the destination
                    C.append(Implies(And(obj[d][t], to[w][t]),
                                     Not(Or(smaller_on_w))))
                # from != to
                C.append(Implies(And(obj[d][t], on[d][w][t]), Not(to[w][t])))
                # update + frame axiom in ONE equivalence
                C.append(on[d][w][t + 1] == If(obj[d][t], to[w][t], on[d][w][t]))

    # ---- symmetry breaking (only sound for shortest plans) -----------------
    if symmetry:
        if k > 0:
            C.append(obj[0][0])                  # first move: smallest disk
        for t in range(k - 1):
            for d in range(n):                   # never move the same disk twice in a row
                C.append(Implies(obj[d][t], Not(obj[d][t + 1])))
    return C, on, obj, to


def extract_plan(model, n, m, k, obj, to):
    plan = []
    for t in range(k):
        d = next((d for d in range(n) if is_true(model.eval(obj[d][t], True))), None)
        if d is None:
            continue                              # idle step
        w = next(w for w in range(m) if is_true(model.eval(to[w][t], True)))
        plan.append((d, w))
    return plan


def check_plan(n, m, plan):
    """Independent simulation of a plan."""
    towers = [list(range(n - 1, -1, -1))] + [[] for _ in range(m - 1)]
    for d, dst in plan:
        src = next(i for i in range(m) if towers[i] and towers[i][-1] == d)
        assert src != dst and (not towers[dst] or towers[dst][-1] > d), "illegal move"
        towers[dst].append(towers[src].pop())
    return len(towers[m - 1]) == n


# ---------------------------------------------------------------------------
# 1) decision problem
def solve_k(n, m, k, idle=False, symmetry=True):
    C, on, obj, to = build(n, m, k, idle, symmetry)
    s = Solver()
    s.add(C)
    r = s.check()
    return r, (extract_plan(s.model(), n, m, k, obj, to) if r == sat else None)


# 2) optimisation, variant A: iterative deepening
def min_steps_iterative(n, m, kmax=1000):
    for k in range(1, kmax + 1):
        r, plan = solve_k(n, m, k)
        if r == sat:
            return k, plan
    return None, None


# 3) optimisation, variant B: bisection on the horizon (idle steps make
#    "solvable in <= K steps" monotone in K)
def min_steps_bisect(n, m, K):
    lo, hi, best = 0, K, None
    while lo < hi:
        mid = (lo + hi) // 2
        r, plan = solve_k(n, m, mid, idle=True)
        if r == sat:
            hi, best = mid, plan
        else:
            lo = mid + 1
    if best is None:
        r, best = solve_k(n, m, lo, idle=True)
    return len(best), best


# 4) optimisation, variant C: Z3 Optimize, minimise the number of real moves
def min_steps_optimize(n, m, K):
    C, on, obj, to = build(n, m, K, idle=True)
    o = Optimize()
    o.add(C)
    cost = Sum([If(Or([obj[d][t] for d in range(n)]), 1, 0) for t in range(K)])
    o.minimize(cost)
    if o.check() == sat:
        plan = extract_plan(o.model(), n, m, K, obj, to)
        return len(plan), plan
    return None, None



if __name__ == "__main__":
    import sys
    M = 3
    NMAX = 7

    print("== feasibility, k = 2^n - 1 ==")
    ns, ks = [], []
    for n in range(3, NMAX + 1):
        k = 2 ** n - 1
        t0 = time.time()
        r, plan = solve_k(n, M, k)
        assert r == sat and len(plan) == k and check_plan(n, M, plan)
        print(f"n={n}  k={k:>4}  {r}  {time.time() - t0:6.2f}s")
        ns.append(n); ks.append(k)

    print("\n== k-1 steps must be unsat (small n only) ==")
    for n in range(3, min(NMAX, 5) + 1):
        t0 = time.time()
        r, _ = solve_k(n, M, 2 ** n - 2)
        print(f"n={n}  k={2 ** n - 2:>4}  {r}  {time.time() - t0:6.2f}s")

    print("\n== optimisation without knowing 2^n-1 ==")
    for n in range(3, min(NMAX, 5) + 1):
        K = 2 ** n + 3                              # any loose upper bound
        for name, f in (("iterative", lambda: min_steps_iterative(n, M)),
                        ("bisect   ", lambda: min_steps_bisect(n, M, K)),
                        ("optimize ", lambda: min_steps_optimize(n, M, K))):
            t0 = time.time()
            best, plan = f()
            assert check_plan(n, M, plan)
            print(f"n={n}  {name}  min={best}  {time.time() - t0:6.2f}s")

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.plot(ns, ks, "o-", label="Z3 minimum")
    plt.plot(ns, [2 ** n - 1 for n in ns], "x--", label=r"$2^n-1$")
    plt.yscale("log"); plt.xlabel("number of disks"); plt.ylabel("number of steps")
    plt.grid(True); plt.legend(); plt.savefig("hanoi_steps.png", dpi=150)
    print("\nsaved hanoi_steps.png")

== feasibility, k = 2^n - 1 ==
n=3  k=   7  sat    0.12s
n=4  k=  15  sat    0.11s
n=5  k=  31  sat    0.61s
n=6  k=  63  sat    1.44s
n=7  k= 127  sat   47.39s

== k-1 steps must be unsat (small n only) ==
n=3  k=   6  unsat    0.03s
n=4  k=  14  unsat    0.09s
n=5  k=  30  unsat    0.33s

== optimisation without knowing 2^n-1 ==
n=3  iterative  min=7    0.12s
n=3  bisect     min=7    0.12s
n=3  optimize   min=7    0.06s
n=4  iterative  min=15    0.73s
n=4  bisect     min=15    0.46s
n=4  optimize   min=15    0.13s
n=5  iterative  min=31    4.91s
n=5  bisect     min=31    1.58s
n=5  optimize   min=31    0.48s

saved hanoi_steps.png
